In [7]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
from torch.utils.data import TensorDataset, DataLoader

In [8]:
# Load embeddings
folder_name = 'extracted-features'
X = np.load(os.path.join(folder_name, "sentencetransformer_embeddings.npy"))
meta = pd.read_csv(os.path.join(folder_name, "sentencetransformer_meta.csv"))

sample_size = min(500000, len(meta))
data_idx = meta.sample(sample_size, random_state=42).index

X_sample = X[data_idx]
y_sent = meta.loc[data_idx, 'sentiment'].values
y_star = meta.loc[data_idx, 'stars'].values - 1  # make labels 0–4

# Train/test split
X_train, X_test, y_sent_train, y_sent_test = train_test_split(
    X_sample, y_sent, test_size=0.1, random_state=42, stratify=y_sent
)
_, _, y_star_train, y_star_test = train_test_split(
    X_sample, y_star, test_size=0.1, random_state=42, stratify=y_star
)


# Normalize embeddings
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert to tensors
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_sent_train_t = torch.tensor(y_sent_train, dtype=torch.long)
y_sent_test_t = torch.tensor(y_sent_test, dtype=torch.long)
y_star_train_t = torch.tensor(y_star_train, dtype=torch.long)
y_star_test_t = torch.tensor(y_star_test, dtype=torch.long)

train_ds = TensorDataset(X_train_t, y_sent_train_t, y_star_train_t)
test_ds = TensorDataset(X_test_t, y_sent_test_t, y_star_test_t)
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

In [9]:
# Define Model
class DeepMultiTaskModel(nn.Module):
    def __init__(self, input_dim=384, shared_dims=[512, 256, 128], dropout=0.3):
        super().__init__()

        layers = []
        prev_dim = input_dim
        for dim in shared_dims:
            layers.extend([
                nn.Linear(prev_dim, dim),
                nn.ReLU(),
                nn.LayerNorm(dim),
                nn.Dropout(dropout)
            ])
            prev_dim = dim
        self.shared = nn.Sequential(*layers)

        # Two heads
        self.sentiment_head = nn.Sequential(
            nn.Linear(shared_dims[-1], 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 3)
        )

        self.star_head = nn.Sequential(
            nn.Linear(shared_dims[-1], 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 5)
        )

    def forward(self, x):
        shared_out = self.shared(x)
        sent_logits = self.sentiment_head(shared_out)
        star_logits = self.star_head(shared_out)
        return sent_logits, star_logits

In [10]:
# Train model
model = MultiTaskModel().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion_sent = nn.CrossEntropyLoss()
criterion_star = nn.CrossEntropyLoss()

EPOCHS = 5
alpha = 0.3  # weight for sentiment loss
beta = 0.7   # weight for star loss

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for Xb, ys_sent, ys_star in train_loader:
        Xb, ys_sent, ys_star = Xb.to(device), ys_sent.to(device), ys_star.to(device)
        optimizer.zero_grad()
        sent_pred, star_pred = model(Xb)
        loss_sent = criterion_sent(sent_pred, ys_sent)
        loss_star = criterion_star(star_pred, ys_star)
        loss = alpha * loss_sent + beta * loss_star
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {total_loss/len(train_loader):.4f}")

Epoch 1/5 - Loss: 1.1301
Epoch 2/5 - Loss: 1.1209
Epoch 3/5 - Loss: 1.1186
Epoch 4/5 - Loss: 1.1167
Epoch 5/5 - Loss: 1.1152


In [11]:
# Evaluate
model.eval()
with torch.no_grad():
    sent_preds, star_preds = [], []
    sent_true, star_true = [], []
    for Xb, ys_sent, ys_star in test_loader:
        Xb = Xb.to(device)
        s_pred, r_pred = model(Xb)
        sent_preds.extend(torch.argmax(s_pred, dim=1).cpu().numpy())
        star_preds.extend(torch.argmax(r_pred, dim=1).cpu().numpy())
        sent_true.extend(ys_sent.numpy())
        star_true.extend(ys_star.numpy())

print("\n=== Sentiment Classification ===")
print(classification_report(sent_true, sent_preds, digits=3))
print(f"Accuracy: {accuracy_score(sent_true, sent_preds):.4f}")

print("\n=== Star Rating Classification ===")
print(classification_report(star_true, star_preds, digits=3))
print(f"Accuracy: {accuracy_score(star_true, star_preds):.4f}")


=== Sentiment Classification ===
              precision    recall  f1-score   support

           0      0.785     0.805     0.795     10246
           1      0.494     0.167     0.249      5654
           2      0.871     0.960     0.913     34100

    accuracy                          0.838     50000
   macro avg      0.717     0.644     0.653     50000
weighted avg      0.811     0.838     0.814     50000

Accuracy: 0.8385

=== Star Rating Classification ===
              precision    recall  f1-score   support

           0      0.000     0.000     0.000      6078
           1      0.000     0.000     0.000      4169
           2      0.000     0.000     0.000      5653
           3      0.167     0.000     0.000     11866
           4      0.445     1.000     0.616     22234

    accuracy                          0.445     50000
   macro avg      0.122     0.200     0.123     50000
weighted avg      0.237     0.445     0.274     50000

Accuracy: 0.4446


/Users/juliasober/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/juliasober/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/juliasober/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
